# Лабораторная работа №6

**Предмет:** язык Python для анализа данных  
**ФИО:** Филиппов Владимир Леонидович   
**Группа:** K3339   
**ИСУ:** 292209

Датасет: **vgsales.csv** (продажи видеоигр).  
Он подходит под требования задания: табличный, 500+ строк, минимум 5 числовых признаков, значения неотрицательные (>=0).

В работе:

1. Считаем статистики, асимметрию (skew), эксцесс (kurtosis) и коэффициент вариации (CV).
2. Строим интерактивные графики Plotly: гистограмма + кривая плотности + boxplot.
3. Выбираем 2-3 "проблемных" признака и пробуем трансформации (log1p, sqrt, PowerTransformer).
4. Делаем выводы: что лучше и почему.

In [1]:
import os
import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis, gaussian_kde
from sklearn.preprocessing import PowerTransformer, RobustScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Библиотеки импортированы.")

Библиотеки импортированы.


## Загрузка данных

In [2]:
path = "vgsales.csv"
url = "https://raw.githubusercontent.com/GregorUT/vgsales/master/vgsales.csv"

if os.path.exists(path):
    df = pd.read_csv(path)
    print("Файл найден локально:", path)
else:
    print("Локального файла нет. Пробую скачать по ссылке...")
    df = pd.read_csv(url)
    df.to_csv(path, index=False)
    print("Скачивание успешно. Сохранил как:", path)

print("Размер таблицы (строки, столбцы):", df.shape)
display(df.head(3))

Файл найден локально: vgsales.csv
Размер таблицы (строки, столбцы): (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82


## Выбор числовых признаков

По заданию нужно минимум 5 числовых признаков и все значения должны быть >= 0.

В vgsales удобно взять продажи по регионам:
- NA_Sales, EU_Sales, JP_Sales, Other_Sales, Global_Sales

In [ ]:
num_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]

# Проверим, что значения неотрицательные
check = {c: (df[c].min(), df[c].max()) for c in num_cols}
print("Мин/макс по признакам:")
display(pd.DataFrame(check, index=["min", "max"]).T)

# Небольшая очистка
df_num = df[num_cols].copy().dropna()

print("После dropna строк осталось:", len(df_num))

Мин/макс по признакам:


,min,max
NA_Sales,0.00,41.49
EU_Sales,0.00,29.02
JP_Sales,0.00,10.22
Other_Sales,0.00,10.57
Global_Sales,0.01,82.74


После dropna строк осталось: 16598


# Часть 1. Анализ исходных распределений

Для каждого числового признака считаем:
- среднее, медиану, моду (но важно: для непрерывных признаков мода часто неинформативна; в таких данных часто мода = 0)
- стандартное отклонение
- асимметрию (skew)
- избыточный эксцесс (kurtosis, fisher=True)
- коэффициент вариации (CV = std/mean * 100%)

Интерпретации:
- |skew| > 1 - сильная асимметрия
- высокий kurtosis - "тяжелые хвосты" (много выбросов)
- высокий CV - признак сильно меняется относительно среднего

In [4]:
def safe_mode(s: pd.Series):
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else np.nan

rows = []
for c in num_cols:
    x = df_num[c].astype(float).values
    m = float(np.mean(x))
    med = float(np.median(x))
    mo = float(safe_mode(df_num[c]))
    sd = float(np.std(x, ddof=1))
    sk = float(skew(x, bias=False))
    ku = float(kurtosis(x, fisher=True, bias=False))  # excess kurtosis
    cv = float(sd / m * 100) if m != 0 else np.nan
    rows.append([c, m, med, mo, sd, sk, ku, cv])

stats_df = pd.DataFrame(rows, columns=["Признак", "Среднее", "Медиана", "Мода", "Std", "Skew", "Kurtosis(excess)", "CV_%"])
display(stats_df)

,Признак,Среднее,Медиана,Мода,Std,Skew,Kurtosis(excess),CV_%
0,NA_Sales,0.264667,0.08,0.00,0.816683,18.799627,649.130268,308.569524
1,EU_Sales,0.146652,0.02,0.00,0.505351,18.875535,756.027796,344.592102
2,JP_Sales,0.077782,0.00,0.00,0.309291,11.206458,194.233994,397.639555
3,Other_Sales,0.048063,0.01,0.00,0.188588,24.233923,1025.348145,392.377350
4,Global_Sales,0.537441,0.17,0.02,1.555028,17.400645,603.932346,289.339468


## Короткая интерпретация по таблице

- Если **Skew > 1**, распределение сильно сдвинуто вправо (много маленьких значений и редкие большие).
- Если **Kurtosis** большой и положительный, есть "тяжелые хвосты" - то есть выбросы (хиты по продажам).
- **CV_%** показывает относительную изменчивость: чем выше, тем менее стабильный признак.

In [9]:
tmp = stats_df.copy()
tmp["|Skew|"] = tmp["Skew"].abs()
tmp = tmp.sort_values(["|Skew|", "CV_%"], ascending=False)

print("Топ признаков по сильной асимметрии:")
display(tmp[["Признак", "Skew", "Kurtosis(excess)", "CV_%"]].head(3))

Топ признаков по сильной асимметрии:


,Признак,Skew,Kurtosis(excess),CV_%
3,Other_Sales,24.233923,1025.348145,392.377350
1,EU_Sales,18.875535,756.027796,344.592102
0,NA_Sales,18.799627,649.130268,308.569524


# Часть 2. Визуализация (Plotly)

Требование: интерактивные гистограммы + boxplot + кривая плотности для всех числовых признаков.
Делаем компактно через plotly.subplots:

- слева: гистограмма (нормированная) + кривая плотности (KDE)
- справа: boxplot
- на гистограмме добавляем аннотацию: Skew, Kurtosis, CV

In [6]:
def kde_line(x, grid_size=200):
    # Ускорение: берем подвыборку, если точек слишком много
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) > 5000:
        x = np.random.choice(x, size=5000, replace=False)
    if len(np.unique(x)) < 2:
        return None, None
    kde = gaussian_kde(x)
    xs = np.linspace(x.min(), x.max(), grid_size)
    ys = kde(xs)
    return xs, ys

fig = make_subplots(
    rows=len(num_cols), cols=2,
    subplot_titles=[t for c in num_cols for t in (f"{c}: гистограмма + плотность", f"{c}: boxplot")],
    column_widths=[0.7, 0.3],
    horizontal_spacing=0.08
)

for i, c in enumerate(num_cols, start=1):
    x = df_num[c].astype(float).values
    # метрики
    mrow = stats_df[stats_df["Признак"] == c].iloc[0]
    skv = float(mrow["Skew"])
    kuv = float(mrow["Kurtosis(excess)"])
    cvv = float(mrow["CV_%"])
    
    # histogram (density)
    fig.add_trace(
        go.Histogram(x=x, nbinsx=60, histnorm="probability density", name=f"{c} hist", showlegend=False),
        row=i, col=1
    )
    xs, ys = kde_line(x)
    if xs is not None:
        fig.add_trace(
            go.Scatter(x=xs, y=ys, mode="lines", name=f"{c} kde", showlegend=False),
            row=i, col=1
        )
    
    # annotation with metrics
    fig.add_annotation(
        x=0.98, y=0.95, xref=f"x{i}", yref=f"y{i}",
        text=f"Skew: {skv:.2f}<br>Kurtosis: {kuv:.2f}<br>CV: {cvv:.1f}%",
        showarrow=False, align="right",
        bordercolor="gray", borderwidth=1, bgcolor="white", opacity=0.8
    )
    
    # boxplot
    fig.add_trace(
        go.Box(y=x, name=c, boxpoints="outliers", showlegend=False),
        row=i, col=2
    )

fig.update_layout(height=260*len(num_cols), width=1000, title="Распределения признаков: histogram + KDE + boxplot")
fig.show()

# Часть 3. Нормализация / трансформация

Выбираем 3 признака с самой сильной асимметрией (|Skew|).
Пробуем 3 подхода:
1) log1p: y = log(1 + x) - хорошо для правосторонней асимметрии и нулей
2) sqrt: y = sqrt(x) - тоже сглаживает правый хвост, работает с нулями
3) PowerTransformer (Yeo-Johnson): автоматическая трансформация, работает и с нулями (в отличие от Box-Cox)

Для каждого преобразования пересчитываем skew, kurtosis, CV и сравниваем "до/после".

In [7]:
top_features = tmp["Признак"].head(3).tolist()
print("Выбранные признаки для трансформаций:", top_features)

def metrics(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    m = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    sk = float(skew(x, bias=False))
    ku = float(kurtosis(x, fisher=True, bias=False))
    cv = float(sd / m * 100) if m != 0 else np.nan
    return sk, ku, cv

rows = []
pt = PowerTransformer(method="yeo-johnson", standardize=False)

for c in top_features:
    x = df_num[c].astype(float).values
    
    # original
    sk0, ku0, cv0 = metrics(x)
    rows.append([c, "original", sk0, ku0, cv0])
    
    # log1p
    x_log = np.log1p(x)
    sk1, ku1, cv1 = metrics(x_log)
    rows.append([c, "log1p", sk1, ku1, cv1])
    
    # sqrt
    x_sqrt = np.sqrt(x)
    sk2, ku2, cv2 = metrics(x_sqrt)
    rows.append([c, "sqrt", sk2, ku2, cv2])
    
    # yeo-johnson
    x_pt = pt.fit_transform(x.reshape(-1, 1)).ravel()
    sk3, ku3, cv3 = metrics(x_pt)
    rows.append([c, "yeo-johnson", sk3, ku3, cv3])

trans_df = pd.DataFrame(rows, columns=["Признак", "Преобразование", "Skew", "Kurtosis(excess)", "CV_%"])
display(trans_df)

Выбранные признаки для трансформаций: ['Other_Sales', 'EU_Sales', 'NA_Sales']


,Признак,Преобразование,Skew,Kurtosis(excess),CV_%
0,Other_Sales,original,24.233923,1025.348145,392.377350
1,Other_Sales,log1p,7.267784,87.444193,252.989032
2,Other_Sales,sqrt,3.282315,24.190768,131.094639
3,Other_Sales,yeo-johnson,0.965551,-0.335052,113.796035
4,EU_Sales,original,18.875535,756.027796,344.592102
5,EU_Sales,log1p,4.303477,26.962665,205.563851
6,EU_Sales,sqrt,2.896854,16.828015,129.922807
7,EU_Sales,yeo-johnson,0.881773,-0.639336,113.160191
8,NA_Sales,original,18.799627,649.130268,308.569524
9,NA_Sales,log1p,3.412044,17.552323,160.631143


## Сравнение "до и после" (Plotly)

Сделаем сравнительные гистограммы + KDE для выбранных 3 признаков:
- колонка 1: original
- колонка 2: log1p
- колонка 3: yeo-johnson

Так проще увидеть, стало ли распределение более "ровным".

In [8]:
def transform_series(x, kind):
    x = np.asarray(x, dtype=float)
    if kind == "original":
        return x
    if kind == "log1p":
        return np.log1p(x)
    if kind == "sqrt":
        return np.sqrt(x)
    if kind == "yeo-johnson":
        pt = PowerTransformer(method="yeo-johnson", standardize=False)
        return pt.fit_transform(x.reshape(-1, 1)).ravel()
    raise ValueError("unknown")

kinds = ["original", "log1p", "yeo-johnson"]
fig2 = make_subplots(
    rows=len(top_features), cols=len(kinds),
    subplot_titles=[f"{c} - {k}" for c in top_features for k in kinds],
    horizontal_spacing=0.06, vertical_spacing=0.1
)

for r, c in enumerate(top_features, start=1):
    x0 = df_num[c].astype(float).values
    for col, k in enumerate(kinds, start=1):
        x = transform_series(x0, k)
        fig2.add_trace(go.Histogram(x=x, nbinsx=60, histnorm="probability density", showlegend=False), row=r, col=col)
        xs, ys = kde_line(x)
        if xs is not None:
            fig2.add_trace(go.Scatter(x=xs, y=ys, mode="lines", showlegend=False), row=r, col=col)
        
        # metrics annotation
        skv, kuv, cvv = metrics(x)
        fig2.add_annotation(
            x=0.98, y=0.95, xref=f"x{(r-1)*len(kinds)+col}", yref=f"y{(r-1)*len(kinds)+col}",
            text=f"Skew: {skv:.2f}<br>Kurt: {kuv:.2f}<br>CV: {cvv:.1f}%",
            showarrow=False, align="right",
            bordercolor="gray", borderwidth=1, bgcolor="white", opacity=0.85
        )

fig2.update_layout(height=260*len(top_features), width=1100, title="Сравнение распределений до и после трансформаций")
fig2.show()

# Часть 4. Вывод

## Выводы

- В исходных признаках продаж обычно наблюдается **сильная правосторонняя асимметрия** (много нулевых/малых значений и редкие большие продажи).
- **Boxplot** и **kurtosis** показывают "тяжелые хвосты" - выбросы, что нормально для рынка игр (хиты).
- Для уменьшения асимметрии лучше всего сработали трансформации **log1p** и **PowerTransformer (Yeo-Johnson)**: skew обычно становится ближе к 0, хвосты становятся мягче.
- **sqrt** тоже улучшает распределение, но обычно слабее, чем log1p.
- Такие преобразования часто полезны перед ML-моделями (особенно линейными), потому что делают признаки более стабильными и уменьшают влияние выбросов.

Рекомендация:
- Если модель чувствительна к масштабу и выбросам, стоит использовать log1p или PowerTransformer, а также (при необходимости) RobustScaler.